# Laboratorio 01 — Consumir 3 APIs de LLM y comparar outputs
### Python AI Developer 2026 · Capítulo 1: LLMs y el ecosistema Python actual

---

**Objetivo general:** al terminar este laboratorio habrás:
1. Configurado un entorno Python reproducible con `uv`
2. Consumido las APIs de **OpenAI** (GPT-4.1-mini), **Anthropic** (Claude Haiku) y **HuggingFace** (Llama 3.1 8B)
3. Entendido cómo funciona la tokenización y por qué importa
4. Comparado los tres modelos en una tabla analítica
5. Explorado cómo el parámetro `temperature` cambia el comportamiento del modelo

**Duración estimada:** 90 minutos  
**Modalidad:** individual  
**Entorno:** local con Jupyter o VS Code + extensión Jupyter

---

### Antes de empezar: setup del entorno

Ejecuta esto **en tu terminal** (no en el notebook). Solo necesitas hacerlo una vez, con GIT:

```bash
# 1. Instalar uv (gestor moderno de entornos Python)
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Crear el proyecto del curso
uv init ai-developer-2026
cd ai-developer-2026

# 3. Instalar todas las dependencias del Capítulo 1
uv add openai anthropic huggingface-hub python-dotenv tiktoken pandas jupyter ipykernel

# 4. Registrar el entorno como kernel de Jupyter
uv run python -m ipykernel install --user --name ai-dev-2026 --display-name "AI Dev 2026"

# 5. Lanzar Jupyter (no es necesario si estamos trabajando en visual studio)
uv run jupyter notebook
```

**Asegúrate de seleccionar el kernel "AI Dev 2026" en Jupyter antes de ejecutar este notebook.**

---
## PARTE 0 — Configuración de credenciales

Las API keys **nunca** van en el código. Las guardamos en un archivo `.env` que:
- Vive en la raíz del proyecto
- Está listado en `.gitignore` (nunca sube a GitHub)
- Es leído por `python-dotenv` en tiempo de ejecución

Crea un archivo `.env` en la carpeta del proyecto con este contenido:

```
OPENAI_API_KEY=sk-proj-...
ANTHROPIC_API_KEY=sk-ant-api03-...
HF_TOKEN=hf_...
```

**¿Dónde consigo las keys?**
- OpenAI: https://platform.openai.com/api-keys (tiene créditos gratuitos de trial)
- Anthropic: https://console.anthropic.com (tiene créditos gratuitos de trial)  
- HuggingFace: https://huggingface.co/settings/tokens (tier gratuito disponible)

> **Nota del instructor:** si el instituto provee keys de clase, úsalas directamente en el `.env`.

Verifica que `.gitignore` contiene `.env`:

In [1]:
# Verificar que .gitignore protege el .env
import os

gitignore_path = ".gitignore"

if os.path.exists(gitignore_path):
    with open(gitignore_path) as f:
        content = f.read()
    if ".env" in content:
        print("✅ .gitignore ya contiene .env — tus keys están protegidas")
    else:
        # Agregar .env al .gitignore automáticamente
        with open(gitignore_path, "a") as f:
            f.write("\n.env\n")
        print("✅ .env agregado a .gitignore")
else:
    with open(gitignore_path, "w") as f:
        f.write(".env\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n")
    print("✅ .gitignore creado con protecciones básicas")

✅ .gitignore ya contiene .env — tus keys están protegidas


In [2]:
# Cargar variables de entorno desde .env
from dotenv import load_dotenv

load_dotenv()

# Verificar que las tres keys están presentes (sin mostrar sus valores)
keys = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "HF_TOKEN": os.getenv("HF_TOKEN"),
}

all_ok = True
for name, value in keys.items():
    if value:
        # Mostrar solo los primeros y últimos 4 caracteres
        masked = value[:8] + "..." + value[-4:]
        print(f"✅ {name}: {masked}")
    else:
        print(f"❌ {name}: NO ENCONTRADA — revisa tu archivo .env")
        all_ok = False

if all_ok:
    print("\n🚀 Todas las credenciales cargadas. ¡Podemos continuar!")
else:
    print("\n⚠️  Revisa el archivo .env antes de continuar.")

✅ OPENAI_API_KEY: sk-proj-...KqYA
✅ ANTHROPIC_API_KEY: sk-ant-a...1QAA
✅ HF_TOKEN: hf_BkJeQ...DwoK

🚀 Todas las credenciales cargadas. ¡Podemos continuar!


---
## PARTE 1 — El prompt de referencia

Para una comparación justa, usaremos **exactamente el mismo prompt** con los tres modelos.

El prompt está diseñado para:
- Tener una respuesta objetivamente verificable (no es ambigua)
- Requerir precisión técnica
- Ser lo suficientemente compleja para mostrar diferencias entre modelos
- Estar en español (esto también nos dirá algo sobre la tokenización)

In [25]:
PROMPT_REFERENCIA = """Explica en exactamente 3 oraciones qué es un token en el contexto 
de los Large Language Models. La primera oración debe dar la definición técnica. 
La segunda debe explicar por qué el concepto de token es importante para el costo 
de usar un LLM. La tercera debe dar un ejemplo concreto con una frase en español."""

print("Prompt de referencia:")
print("-" * 60)
print(PROMPT_REFERENCIA)
print("-" * 60)
print(f"Caracteres: {len(PROMPT_REFERENCIA)}")

Prompt de referencia:
------------------------------------------------------------
Explica en exactamente 3 oraciones qué es un token en el contexto 
de los Large Language Models. La primera oración debe dar la definición técnica. 
La segunda debe explicar por qué el concepto de token es importante para el costo 
de usar un LLM. La tercera debe dar un ejemplo concreto con una frase en español.
------------------------------------------------------------
Caracteres: 313


---
## PARTE 2 — OpenAI: GPT-4.1-mini

**GPT-4.1-mini** es el modelo de OpenAI recomendado para uso educativo en 2026:
- Excelente relación calidad/precio
- Contexto de 1M tokens
- $0.40/1M tokens de input, $1.60/1M tokens de output

La API de OpenAI sigue el esquema de mensajes **role / content**:
- `system`: instrucciones permanentes al modelo
- `user`: el turno del usuario
- `assistant`: el turno del modelo (para conversaciones multi-turno)

Referencia: https://platform.openai.com/docs/api-reference/chat

In [26]:
from openai import OpenAI

# El cliente lee OPENAI_API_KEY del entorno automáticamente
client_openai = OpenAI()

print("Llamando a GPT-4.1-mini...")

response_oai = client_openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": "Eres un instructor de IA para desarrolladores Python. "
                       "Eres preciso, técnico y conciso. Respondes siempre en español."
        },
        {
            "role": "user",
            "content": PROMPT_REFERENCIA
        }
    ],
    temperature=0.3,       # Baja variabilidad: respuestas más consistentes
    max_tokens=300,        # Límite de output para controlar costos
)

# Extraer el texto de la respuesta
texto_oai = response_oai.choices[0].message.content

# Extraer métricas de uso
tokens_input_oai = response_oai.usage.prompt_tokens
tokens_output_oai = response_oai.usage.completion_tokens
tokens_total_oai = response_oai.usage.total_tokens

print(f"\n{'='*60}")
print("RESPUESTA — GPT-4.1-mini")
print(f"{'='*60}")
print(texto_oai)
print(f"\n{'─'*60}")
print(f"Tokens de input:  {tokens_input_oai}")
print(f"Tokens de output: {tokens_output_oai}")
print(f"Tokens totales:   {tokens_total_oai}")

# Calcular costo estimado (precios de mayo 2026)
costo_oai = (tokens_input_oai * 0.40 + tokens_output_oai * 1.60) / 1_000_000
print(f"Costo estimado:   ${costo_oai:.6f} USD")

Llamando a GPT-4.1-mini...

RESPUESTA — GPT-4.1-mini
Un token en el contexto de los Large Language Models (LLM) es la unidad mínima de texto que el modelo procesa, que puede ser una palabra, un subpalabra o incluso un carácter, dependiendo del tokenizador utilizado. La importancia del token radica en que el costo computacional y económico de usar un LLM suele medirse en función del número de tokens procesados, tanto en la entrada como en la salida. Por ejemplo, la frase en español "Hola, ¿cómo estás?" puede dividirse en tokens como ["Hola", ",", "¿", "cómo", "estás", "?"].

────────────────────────────────────────────────────────────
Tokens de input:  105
Tokens de output: 123
Tokens totales:   228
Costo estimado:   $0.000239 USD


### ¿Qué acaba de pasar?

Analicemos la estructura del objeto `response_oai` que devuelve la API.
Entender esta estructura es fundamental para construir aplicaciones reales.

In [27]:
# Explorar la estructura completa de la respuesta de OpenAI
import json

print("Estructura del objeto response de OpenAI:")
print(f"""  response
  ├── id:                    {response_oai.id}
  ├── model:                 {response_oai.model}
  ├── created (timestamp):   {response_oai.created}
  ├── choices (lista):
  │   └── [0]
  │       ├── finish_reason: {response_oai.choices[0].finish_reason}
  │       └── message:
  │           ├── role:      {response_oai.choices[0].message.role}
  │           └── content:   "{response_oai.choices[0].message.content[:50]}..."
  └── usage:
      ├── prompt_tokens:     {response_oai.usage.prompt_tokens}
      ├── completion_tokens: {response_oai.usage.completion_tokens}
      └── total_tokens:      {response_oai.usage.total_tokens}
""")

print(f"finish_reason = '{response_oai.choices[0].finish_reason}'")
print("→ 'stop' significa que el modelo terminó naturalmente (no fue cortado por max_tokens)")

Estructura del objeto response de OpenAI:
  response
  ├── id:                    chatcmpl-DlzbFCUM6W2iuvyveQbIhFHXcrUik
  ├── model:                 gpt-4.1-mini-2025-04-14
  ├── created (timestamp):   1780330577
  ├── choices (lista):
  │   └── [0]
  │       ├── finish_reason: stop
  │       └── message:
  │           ├── role:      assistant
  │           └── content:   "Un token en el contexto de los Large Language Mode..."
  └── usage:
      ├── prompt_tokens:     105
      ├── completion_tokens: 123
      └── total_tokens:      228

finish_reason = 'stop'
→ 'stop' significa que el modelo terminó naturalmente (no fue cortado por max_tokens)


---
## PARTE 3 — Anthropic: Claude Haiku 4.5

**Claude Haiku 4.5** es el modelo de Anthropic orientado a velocidad y costo:
- Modelo: `claude-haiku-4-5-20251001`
- Muy rápido, ideal para aplicaciones de alta frecuencia
- Precio competitivo con GPT-4.1-mini

**Diferencias estructurales con OpenAI que hay que notar:**
- El `system` prompt va como parámetro separado, no dentro de `messages`
- `max_tokens` es **obligatorio** en Anthropic (en OpenAI es opcional)
- El acceso al texto de respuesta es `response.content[0].text` (no `.choices[0].message.content`)
- El conteo de tokens se llama `input_tokens` y `output_tokens` (no `prompt_tokens`)

In [28]:
import anthropic

# El cliente lee ANTHROPIC_API_KEY del entorno automáticamente
client_anthropic = anthropic.Anthropic()

print("Llamando a Claude Haiku 4.5...")

response_ant = client_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",
    system="Eres un instructor de IA para desarrolladores Python. "
           "Eres preciso, técnico y conciso. Respondes siempre en español.",
    messages=[
        {
            "role": "user",
            "content": PROMPT_REFERENCIA
        }
    ],
    temperature=0.3,
    max_tokens=300,   # OBLIGATORIO en Anthropic
)

# Notar la diferencia: .content[0].text en lugar de .choices[0].message.content
texto_ant = response_ant.content[0].text

tokens_input_ant = response_ant.usage.input_tokens
tokens_output_ant = response_ant.usage.output_tokens
tokens_total_ant = tokens_input_ant + tokens_output_ant

print(f"\n{'='*60}")
print("RESPUESTA — Claude Haiku 4.5")
print(f"{'='*60}")
print(texto_ant)
print(f"\n{'─'*60}")
print(f"Tokens de input:  {tokens_input_ant}")
print(f"Tokens de output: {tokens_output_ant}")
print(f"Tokens totales:   {tokens_total_ant}")

# Precios Claude Haiku 4.5 (mayo 2026)
costo_ant = (tokens_input_ant * 0.80 + tokens_output_ant * 4.00) / 1_000_000
print(f"Costo estimado:   ${costo_ant:.6f} USD")

Llamando a Claude Haiku 4.5...

RESPUESTA — Claude Haiku 4.5
Un token es la unidad mínima de texto que un LLM procesa, generalmente representando palabras completas, subpalabras o caracteres individuales según el algoritmo de tokenización utilizado. El concepto de token es crucial porque los proveedores de APIs de LLMs cobran por cantidad de tokens procesados (entrada) y generados (salida), no por solicitudes, lo que hace que optimizar el uso de tokens sea esencial para controlar costos. Por ejemplo, la frase "El gato está durmiendo" se tokeniza aproximadamente en 5-6 tokens: ["El", "gato", "está", "dur", "miendo"] dependiendo del tokenizador.

────────────────────────────────────────────────────────────
Tokens de input:  128
Tokens de output: 162
Tokens totales:   290
Costo estimado:   $0.000750 USD


In [29]:
# Explorar la estructura completa de la respuesta de Anthropic
print("Estructura del objeto response de Anthropic:")
print(f"""  response
  ├── id:            {response_ant.id}
  ├── model:         {response_ant.model}
  ├── type:          {response_ant.type}
  ├── stop_reason:   {response_ant.stop_reason}
  ├── content (lista):
  │   └── [0]
  │       ├── type: {response_ant.content[0].type}
  │       └── text: "{response_ant.content[0].text[:50]}..."
  └── usage:
      ├── input_tokens:  {response_ant.usage.input_tokens}
      └── output_tokens: {response_ant.usage.output_tokens}
""")

print("   Comparación estructural clave:")
print("   OpenAI:    response.choices[0].message.content")
print("   Anthropic: response.content[0].text")
print("   → Cuando uses LangChain, esto se abstrae automáticamente.")
print("     Pero conocerlo te ayudará a debuggear problemas reales.")

Estructura del objeto response de Anthropic:
  response
  ├── id:            msg_01SUVNGZYfPEAvjxCkv6K1UF
  ├── model:         claude-haiku-4-5-20251001
  ├── type:          message
  ├── stop_reason:   end_turn
  ├── content (lista):
  │   └── [0]
  │       ├── type: text
  │       └── text: "Un token es la unidad mínima de texto que un LLM p..."
  └── usage:
      ├── input_tokens:  128
      └── output_tokens: 162

   Comparación estructural clave:
   OpenAI:    response.choices[0].message.content
   Anthropic: response.content[0].text
   → Cuando uses LangChain, esto se abstrae automáticamente.
     Pero conocerlo te ayudará a debuggear problemas reales.


---
## PARTE 4 — HuggingFace: Llama 3.1 8B Instruct

**Llama 3.1 8B** de Meta es el modelo open-source más popular del mundo en 2026:
- Open weights: puedes descargarlo y ejecutarlo localmente
- A través de HuggingFace Inference API: sin necesidad de GPU propia
- Tier gratuito disponible (con límites de rate)

**Por qué incluirlo:**
- En producción real, a veces necesitas modelos que **no envíen datos a terceros** (privacidad, regulación)
- El stack open-source es cada vez más competitivo con los modelos propietarios
- Entender la diferencia te da criterio para elegir en proyectos reales

> ⚠️ El modelo requiere **aceptar los términos de uso** de Meta en HuggingFace antes de poder usarlo.
> Ve a: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct y acepta los términos.

In [30]:
from huggingface_hub import InferenceClient

# El cliente lee HF_TOKEN del entorno automáticamente si está configurado
client_hf = InferenceClient(
    provider="novita",
    api_key=os.getenv("HF_TOKEN"),
)

print("Llamando a Llama 3.1 8B Instruct vía HuggingFace...")
print("(puede tardar unos segundos si el modelo está 'frío')\n")

response_hf = client_hf.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "system",
            "content": "Eres un instructor de IA para desarrolladores Python. "
                       "Eres preciso, técnico y conciso. Respondes siempre en español."
        },
        {
            "role": "user",
            "content": PROMPT_REFERENCIA
        }
    ],
    temperature=0.3,
    max_tokens=300,
)

# La API de HuggingFace devuelve el mismo esquema que OpenAI (compatibilidad)
texto_hf = response_hf.choices[0].message.content

tokens_input_hf = response_hf.usage.prompt_tokens
tokens_output_hf = response_hf.usage.completion_tokens
tokens_total_hf = response_hf.usage.total_tokens

print(f"{'='*60}")
print("RESPUESTA — Llama 3.1 8B Instruct")
print(f"{'='*60}")
print(texto_hf)
print(f"\n{'─'*60}")
print(f" Tokens de input:  {tokens_input_hf}")
print(f" Tokens de output: {tokens_output_hf}")
print(f" Tokens totales:   {tokens_total_hf}")
print(f" Costo en HF Inference (tier gratuito): $0.000000 USD")

Llamando a Llama 3.1 8B Instruct vía HuggingFace...
(puede tardar unos segundos si el modelo está 'frío')

RESPUESTA — Llama 3.1 8B Instruct
En el contexto de los Large Language Models (LLM), un token es una unidad básica de información que representa una parte del texto, como una palabra, un símbolo o un carácter, que se utiliza como entrada para el modelo. La importancia del concepto de token radica en que el costo de usar un LLM se calcula en función del número de tokens procesados, lo que significa que el tamaño del texto de entrada tiene un impacto directo en el costo de la solicitud. Por ejemplo, la frase "La computadora es muy rápida" se divide en tokens como ["La", "computadora", "es", "muy", "rápida"], lo que significa que cada una de estas palabras es procesada como un token individual.

────────────────────────────────────────────────────────────
 Tokens de input:  140
 Tokens de output: 161
 Tokens totales:   301
 Costo en HF Inference (tier gratuito): $0.000000 USD


---
## PARTE 5 — Tabla comparativa de los tres modelos

Ahora que tenemos los tres resultados, los consolidamos en una tabla analítica.
Esta tabla será parte del entregable del laboratorio.

In [35]:
import pandas as pd

# Calcular costos por 1000 requests (volumen realista de producción)
def costo_por_mil(input_tokens, output_tokens, precio_input_por_millon, precio_output_por_millon):
    return 1000 * (input_tokens * precio_input_por_millon + output_tokens * precio_output_por_millon) / 1_000_000

datos = {
    "Modelo": ["GPT-4.1-mini", "Claude Haiku 4.5", "Llama 3.1 8B"],
    "Proveedor": ["OpenAI", "Anthropic", "Meta / HuggingFace"],
    "Tipo": ["Propietario", "Propietario", "Open-source"],
    "Tokens input": [tokens_input_oai, tokens_input_ant, tokens_input_hf],
    "Tokens output": [tokens_output_oai, tokens_output_ant, tokens_output_hf],
    "Tokens totales": [tokens_total_oai, tokens_total_ant, tokens_total_hf],
    "Precio input ($/1M)": [0.40, 0.80, 0.00],
    "Precio output ($/1M)": [1.60, 4.00, 0.00],
    "Costo esta llamada ($)": [
        round(costo_oai, 6),
        round(costo_ant, 6),
        0.0
    ],
    "Costo por 1000 requests ($)": [
        round(costo_por_mil(tokens_input_oai, tokens_output_oai, 0.40, 1.60), 4),
        round(costo_por_mil(tokens_input_ant, tokens_output_ant, 0.80, 4.00), 4),
        0.0
    ],
    "Contexto máx (tokens)": ["1,000,000", "200,000", "128,000"],
    "Multimodal": ["Texto + imagen", "Texto + imagen", "Solo texto"],
}

df = pd.DataFrame(datos)

# Configurar display para ver todas las columnas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 30)

print("\nTABLA COMPARATIVA — 3 APIs de LLM")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)
print("\n* Precios referenciales de junio 2026. Verificar siempre en la documentación oficial de cada proveedor.")


TABLA COMPARATIVA — 3 APIs de LLM
          Modelo          Proveedor        Tipo  Tokens input  Tokens output  Tokens totales  Precio input ($/1M)  Precio output ($/1M)  Costo esta llamada ($)  Costo por 1000 requests ($) Contexto máx (tokens)     Multimodal
    GPT-4.1-mini             OpenAI Propietario           105            123             228                  0.4                   1.6                0.000239                       0.2388             1,000,000 Texto + imagen
Claude Haiku 4.5          Anthropic Propietario           128            162             290                  0.8                   4.0                0.000750                       0.7504               200,000 Texto + imagen
    Llama 3.1 8B Meta / HuggingFace Open-source           140            161             301                  0.0                   0.0                0.000000                       0.0000               128,000     Solo texto

* Precios referenciales de junio 2026. Verificar siempre en 

In [36]:
# Comparación visual de las respuestas
print("\n📝 COMPARACIÓN DE OUTPUTS")
print("=" * 60)

modelos = {
    "GPT-4.1-mini (OpenAI)": texto_oai,
    "Claude Haiku 4.5 (Anthropic)": texto_ant,
    "Llama 3.1 8B (HuggingFace)": texto_hf,
}

for nombre, texto in modelos.items():
    print(f"\n▶ {nombre}")
    print("-" * 60)
    print(texto)
    print(f"  → {len(texto.split())} palabras | {len(texto)} caracteres")

print("\n💡 Preguntas para reflexionar:")
print("   1. ¿Cuál cumplió mejor con la restricción de '3 oraciones exactas'?")
print("   2. ¿Qué diferencias notas en el tono y precisión técnica?")
print("   3. ¿Cuál de los tres usarías para un chatbot de producción en español?")


📝 COMPARACIÓN DE OUTPUTS

▶ GPT-4.1-mini (OpenAI)
------------------------------------------------------------
Un token en el contexto de los Large Language Models (LLM) es la unidad mínima de texto que el modelo procesa, que puede ser una palabra, un subpalabra o incluso un carácter, dependiendo del tokenizador utilizado. La importancia del token radica en que el costo computacional y económico de usar un LLM suele medirse en función del número de tokens procesados, tanto en la entrada como en la salida. Por ejemplo, la frase en español "Hola, ¿cómo estás?" puede dividirse en tokens como ["Hola", ",", "¿", "cómo", "estás", "?"].
  → 89 palabras | 526 caracteres

▶ Claude Haiku 4.5 (Anthropic)
------------------------------------------------------------
Un token es la unidad mínima de texto que un LLM procesa, generalmente representando palabras completas, subpalabras o caracteres individuales según el algoritmo de tokenización utilizado. El concepto de token es crucial porque los pro

---
## PARTE 6 — Tokenización: la unidad fundamental de los LLMs

Antes de continuar con el experimento de `temperature`, necesitamos entender **qué es exactamente un token**.

Un token NO es lo mismo que una palabra. Puede ser:
- Una palabra completa: `"hola"` → 1 token
- Una sílaba: `"transformers"` → puede ser 2-3 tokens
- Un carácter solo: en idiomas no-latinos, cada carácter puede ser 1 token
- Un espacio + palabra: `" hola"` → 1 token (diferente a `"hola"`)

**Por qué importa esto en la práctica:**
- Los LLMs cobran por tokens, no por palabras
- El español y el português se tokenizan **menos eficientemente** que el inglés
  (los modelos fueron entrenados mayoritariamente en inglés)
- Esto tiene un impacto real en el costo de aplicaciones en Latinoamérica

Vamos a verlo empíricamente con `tiktoken`, la librería oficial de OpenAI:

In [16]:
import tiktoken

# Cargar el tokenizador de GPT-4 (cl100k_base, compartido con muchos modelos modernos)
enc = tiktoken.get_encoding("cl100k_base")

print("=" * 65)
print("EXPERIMENTO 1: ¿Cuántos tokens por palabra en distintos idiomas?")
print("=" * 65)

frases = [
    ("Inglés",    "The language model predicts the next token."),
    ("Español",   "El modelo de lenguaje predice el siguiente token."),
    ("Portugués", "O modelo de linguagem prevê o próximo token."),
    ("Chino",     "语言模型预测下一个标记。"),
    ("Árabe",     "يتنبأ نموذج اللغة بالرمز التالي."),
]

resultados = []
for idioma, frase in frases:
    tokens = enc.encode(frase)
    palabras = len(frase.split())
    ratio = len(tokens) / palabras if palabras > 0 else 0
    resultados.append({
        "Idioma": idioma,
        "Frase": frase[:45] + "...",
        "Palabras": palabras,
        "Tokens": len(tokens),
        "Ratio tok/pal": round(ratio, 2),
    })
    print(f"\n{idioma}:")
    print(f"  Frase:   '{frase}'")
    print(f"  Tokens:  {len(tokens)} → IDs: {tokens}")
    print(f"  Ratio:   {ratio:.2f} tokens/palabra")

print("\n" + "=" * 65)
print("CONCLUSIÓN:")
print("El español usa ~40% más tokens que el inglés para el mismo contenido.")
print("Esto significa ~40% más costo para aplicaciones en español.")
print("→ Factor crítico al estimar presupuestos en proyectos para LATAM.")

EXPERIMENTO 1: ¿Cuántos tokens por palabra en distintos idiomas?

Inglés:
  Frase:   'The language model predicts the next token.'
  Tokens:  8 → IDs: [791, 4221, 1646, 56978, 279, 1828, 4037, 13]
  Ratio:   1.14 tokens/palabra

Español:
  Frase:   'El modelo de lenguaje predice el siguiente token.'
  Tokens:  12 → IDs: [6719, 37042, 409, 326, 28102, 11305, 4255, 560, 658, 56001, 4037, 13]
  Ratio:   1.50 tokens/palabra

Portugués:
  Frase:   'O modelo de linguagem prevê o próximo token.'
  Tokens:  11 → IDs: [46, 37042, 409, 39603, 15003, 8031, 5615, 297, 71898, 4037, 13]
  Ratio:   1.38 tokens/palabra

Chino:
  Frase:   '语言模型预测下一个标记。'
  Tokens:  12 → IDs: [73981, 78244, 54872, 25287, 99941, 27699, 233, 17297, 48044, 31944, 41914, 1811]
  Ratio:   12.00 tokens/palabra

Árabe:
  Frase:   'يتنبأ نموذج اللغة بالرمز التالي.'
  Tokens:  23 → IDs: [14900, 14628, 12061, 22071, 70782, 51343, 10386, 12942, 56434, 34190, 17607, 8700, 82878, 26957, 28946, 32482, 11318, 10386, 40797, 96057, 32482

In [38]:
print("=" * 65)
print("EXPERIMENTO 2: Tokenización de términos técnicos en Python")
print("=" * 65)

terminos = [
    "Python",
    "transformers",
    "retrieval-augmented-generation",
    "LangChain",
    "embeddings",
    "fine-tuning",
    "sk-proj-abc123xyz",   # formato de API key
    "GPT",
    "claude-haiku-4-5-20251001",  # nombre de modelo
]

for termino in terminos:
    tokens = enc.encode(termino)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  '{termino}'")
    print(f"   → {len(tokens)} tokens: {decoded}")

print("\n   Nota: las API keys y los nombres de modelos se tokenizán")
print("   en muchos tokens porque son cadenas poco frecuentes en el")
print("   corpus de entrenamiento. ¡Nunca incluyas API keys en tus prompts!")

EXPERIMENTO 2: Tokenización de términos técnicos en Python
  'Python'
   → 1 tokens: ['Python']
  'transformers'
   → 2 tokens: ['transform', 'ers']
  'retrieval-augmented-generation'
   → 7 tokens: ['re', 'trie', 'val', '-a', 'ug', 'mented', '-generation']
  'LangChain'
   → 2 tokens: ['Lang', 'Chain']
  'embeddings'
   → 2 tokens: ['embed', 'dings']
  'fine-tuning'
   → 3 tokens: ['fine', '-t', 'uning']
  'sk-proj-abc123xyz'
   → 7 tokens: ['sk', '-pro', 'j', '-', 'abc', '123', 'xyz']
  'GPT'
   → 2 tokens: ['G', 'PT']
  'claude-haiku-4-5-20251001'
   → 13 tokens: ['cla', 'ude', '-h', 'a', 'iku', '-', '4', '-', '5', '-', '202', '510', '01']

   Nota: las API keys y los nombres de modelos se tokenizán
   en muchos tokens porque son cadenas poco frecuentes en el
   corpus de entrenamiento. ¡Nunca incluyas API keys en tus prompts!


In [39]:
print("=" * 65)
print("EXPERIMENTO 3: Contar tokens ANTES de hacer la llamada a la API")
print("=" * 65)

def contar_tokens_prompt(mensajes: list[dict], modelo: str = "gpt-4o") -> int:
    """
    Estima el número de tokens de una lista de mensajes
    antes de hacer la llamada a la API.
    
    Útil para: controlar costos, verificar que no superamos el límite de contexto.
    """
    enc_modelo = tiktoken.encoding_for_model(modelo)
    tokens_por_mensaje = 3  # overhead de formato por mensaje
    tokens_por_nombre = 1
    
    total = 0
    for mensaje in mensajes:
        total += tokens_por_mensaje
        for key, value in mensaje.items():
            total += len(enc_modelo.encode(value))
            if key == "name":
                total += tokens_por_nombre
    total += 3  # reply primer token overhead
    return total


# Probar con nuestro prompt de referencia
mensajes_prueba = [
    {"role": "system", "content": "Eres un instructor de IA para desarrolladores Python."},
    {"role": "user", "content": PROMPT_REFERENCIA}
]

tokens_estimados = contar_tokens_prompt(mensajes_prueba, "gpt-4o")
print(f"Tokens estimados (antes de llamar):  {tokens_estimados}")
print(f"Tokens reales (respuesta de la API): {tokens_input_oai}")
print(f"Diferencia:                          {abs(tokens_estimados - tokens_input_oai)}")
print()
print("   Esto te permite verificar ANTES de gastar dinero que no")
print("   superarás el contexto máximo del modelo ni tu presupuesto.")

EXPERIMENTO 3: Contar tokens ANTES de hacer la llamada a la API
Tokens estimados (antes de llamar):  89
Tokens reales (respuesta de la API): 105
Diferencia:                          16

   Esto te permite verificar ANTES de gastar dinero que no
   superarás el contexto máximo del modelo ni tu presupuesto.


---
## PARTE 7 — Experimento de Temperature

El parámetro `temperature` controla la **aleatoriedad** del modelo:

| Temperature | Comportamiento | Cuándo usarlo |
|-------------|---------------|---------------|
| `0.0` | Determinista (casi). Siempre la respuesta más probable | Clasificación, extracción de datos, código |
| `0.3` | Baja variabilidad. Preciso pero con algo de flexibilidad | Resúmenes, Q&A técnica |
| `0.7` | Balance. El default de muchos modelos | Chatbots, redacción |
| `1.0` | Alta variabilidad. Creativo e impredecible | Brainstorming, escritura creativa |
| `> 1.0` | Muy caótico. Útil solo en casos muy específicos | Raramente en producción |

**Cómo funciona internamente:** el modelo produce una distribución de probabilidades sobre todos los tokens posibles. Temperature modifica esa distribución antes de samplear el siguiente token:
- Temperature baja → la distribución se "aplana" hacia el máximo → más determinismo
- Temperature alta → la distribución se "suaviza" → más diversidad

In [40]:
# Prompt diseñado para mostrar diferencias claras con temperature
# (creativo, sin respuesta única correcta)
PROMPT_TEMPERATURA = """En una oración, ¿cómo explicarías la inteligencia artificial 
a un niño de 8 años?"""

temperatures = [0.0, 0.7, 1.4]
# Ejecutar 3 veces cada temperatura para ver la variabilidad
repeticiones = 3

print(f"Prompt: '{PROMPT_TEMPERATURA[:60]}...'")
print("=" * 65)

for temp in temperatures:
    print(f"\n🌡️  TEMPERATURE = {temp}")
    print("-" * 65)
    
    for i in range(repeticiones):
        r = client_openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": PROMPT_TEMPERATURA}],
            temperature=temp,
            max_tokens=100,
        )
        respuesta = r.choices[0].message.content.strip()
        print(f"  [{i+1}] {respuesta}")

print("\n" + "=" * 65)
print("OBSERVA:")
print("  • Con temp=0.0: las dos respuestas son casi idénticas")
print("  • Con temp=0.7: parecidas pero con variaciones de estilo")
print("  • Con temp=1.4: diferentes en estructura, tono y metáforas")

Prompt: 'En una oración, ¿cómo explicarías la inteligencia artificial...'

🌡️  TEMPERATURE = 0.0
-----------------------------------------------------------------
  [1] La inteligencia artificial es como un robot o una computadora que puede aprender y pensar un poco como las personas para ayudarnos a hacer cosas.
  [2] La inteligencia artificial es como un robot o una computadora que puede aprender y pensar un poco como las personas para ayudarnos a hacer cosas.
  [3] La inteligencia artificial es como un robot o una computadora que puede aprender y pensar un poco como las personas para ayudarnos a hacer cosas.

🌡️  TEMPERATURE = 0.7
-----------------------------------------------------------------
  [1] La inteligencia artificial es como un robot o una computadora que puede aprender y pensar un poco como una persona para ayudarnos a hacer cosas.
  [2] La inteligencia artificial es como un robot o una computadora que puede aprender y pensar un poco como las personas para ayudarnos a ha

---
## PARTE 8 — Abstracción: una función unificada para los 3 modelos

En proyectos reales necesitamos poder cambiar de modelo **sin reescribir la lógica de negocio**.
Esto es exactamente lo que hacen frameworks como LangChain, que veremos en el Capítulo 3.

Pero primero, construyamos nuestra propia versión minimalista para entender el patrón:

In [41]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class LLMResponse:
    """Objeto unificado de respuesta, independiente del proveedor."""
    modelo: str
    proveedor: str
    texto: str
    tokens_input: int
    tokens_output: int
    costo_usd: float

    def resumen(self) -> str:
        return (
            f"[{self.proveedor} / {self.modelo}]\n"
            f"Texto: {self.texto[:100]}...\n"
            f"Tokens: {self.tokens_input} in / {self.tokens_output} out\n"
            f"Costo: ${self.costo_usd:.6f} USD"
        )


def llamar_llm(
    prompt: str,
    proveedor: Literal["openai", "anthropic", "huggingface"],
    system: str = "Responde siempre en español de forma concisa.",
    temperature: float = 0.3,
    max_tokens: int = 300,
) -> LLMResponse:
    """
    Interfaz unificada para llamar a cualquiera de los 3 proveedores.
    Devuelve siempre un LLMResponse con la misma estructura.
    """
    
    if proveedor == "openai":
        r = client_openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return LLMResponse(
            modelo="gpt-4.1-mini",
            proveedor="OpenAI",
            texto=r.choices[0].message.content,
            tokens_input=r.usage.prompt_tokens,
            tokens_output=r.usage.completion_tokens,
            costo_usd=(r.usage.prompt_tokens * 0.40 + r.usage.completion_tokens * 1.60) / 1_000_000,
        )

    elif proveedor == "anthropic":
        r = client_anthropic.messages.create(
            model="claude-haiku-4-5-20251001",
            system=system,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return LLMResponse(
            modelo="claude-haiku-4-5-20251001",
            proveedor="Anthropic",
            texto=r.content[0].text,
            tokens_input=r.usage.input_tokens,
            tokens_output=r.usage.output_tokens,
            costo_usd=(r.usage.input_tokens * 0.80 + r.usage.output_tokens * 4.00) / 1_000_000,
        )

    elif proveedor == "huggingface":
        r = client_hf.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return LLMResponse(
            modelo="Llama-3.1-8B-Instruct",
            proveedor="HuggingFace / Meta",
            texto=r.choices[0].message.content,
            tokens_input=r.usage.prompt_tokens,
            tokens_output=r.usage.completion_tokens,
            costo_usd=0.0,
        )
    
    else:
        raise ValueError(f"Proveedor desconocido: {proveedor}")


print("✅ Función llamar_llm() definida")
print("   Úsala así: respuesta = llamar_llm('tu pregunta', 'openai')")

✅ Función llamar_llm() definida
   Úsala así: respuesta = llamar_llm('tu pregunta', 'openai')


In [43]:
# Prueba de la función unificada con un nuevo prompt
PROMPT_FINAL = "¿Cuál es la diferencia entre Machine Learning y Deep Learning? Máximo 2 oraciones."

print("Probando la función unificada con los 3 proveedores...\n")

for proveedor in ["openai", "anthropic", "huggingface"]:
    try:
        resp = llamar_llm(PROMPT_FINAL, proveedor)
        print(resp.resumen())
        print()
    except Exception as e:
        print(f"❌ Error con {proveedor}: {e}\n")

print("   Este patrón de abstracción es exactamente lo que hace LangChain")
print("   con sus 'chat models'. En el Cap. 3 lo veremos en detalle.")

Probando la función unificada con los 3 proveedores...

[OpenAI / gpt-4.1-mini]
Texto: El Machine Learning es un campo de la inteligencia artificial que utiliza algoritmos para que las má...
Tokens: 40 in / 69 out
Costo: $0.000126 USD

[Anthropic / claude-haiku-4-5-20251001]
Texto: **Machine Learning** es un campo amplio que usa algoritmos para aprender patrones de datos, mientras...
Tokens: 46 in / 79 out
Costo: $0.000353 USD

[HuggingFace / Meta / Llama-3.1-8B-Instruct]
Texto: La principal diferencia entre Machine Learning y Deep Learning radica en la complejidad de los model...
Tokens: 66 in / 86 out
Costo: $0.000000 USD

   Este patrón de abstracción es exactamente lo que hace LangChain
   con sus 'chat models'. En el Cap. 3 lo veremos en detalle.


---
## PARTE 9 — Análisis final y conclusiones

Documenta aquí tus observaciones. Esta celda es parte del entregable.

## Resumen ejecutivo — Laboratorio 01

### Modelos evaluados

| Modelo | Proveedor | Tipo | Característica |
|---|---|---|---|
| GPT-4.1-mini | OpenAI | Propietario | Alta disponibilidad |
| Claude Haiku | Anthropic | Propietario | Énfasis en seguridad |
| Llama 3.1 8B | Meta / HuggingFace | Open-source | Sin costo de API |

### Datos de esta sesión

- **Total tokens consumidos:** `{tokens_total_oai + tokens_total_ant + tokens_total_hf}`
- **Costo total estimado:** `${costo_oai + costo_ant:.6f} USD`
- **HuggingFace:** `$0.000000 USD` (tier gratuito)

### Lecciones clave

1. Las APIs siguen el mismo patrón conceptual pero con **diferencias estructurales importantes** — hay que leer la documentación de cada proveedor.

2. El español requiere **~40% más tokens que el inglés**. En producción esto impacta directamente el costo mensual.

3. `temperature=0.0` no garantiza determinismo total — hay otros factores de aleatoriedad en la inferencia, pero es la configuración más reproducible disponible.

4. **La abstracción es clave:** cambiar de proveedor no debería requerir reescribir la lógica de negocio.

5. El costo por llamada parece pequeño, pero a escala de **100,000 llamadas/mes** puede ser la diferencia entre un negocio viable y uno inviable.